# Validación RegionFunctionLDA

Comprueba paso a paso que el módulo LDA funciona correctamente:
1. Carga de datos OD
2. Preparación temporal (slots y daytype)
3. Construcción de documentos
4. Matriz documento-término
5. Ajuste LDA y Theta
6. Inspección de tópicos

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

sys.path.insert(0, '.')
from region_function_lda import RegionFunctionLDA

MATRICES_DIR = Path('../../cubo_izquierdo/matrices')

# Datos de febrero 2025: 20 laborables + 8 finde de semana = 28 días
# No hay festivos entre semana en Madrid en este período →
# holiday_dates vacío (sáb/dom ya se clasifican WE automáticamente en _prepare)
OD_FOLDERS = [
    'validaciones_3_7_febrero',      # lun–vie  3– 7 feb  (5 laborables)
    'validaciones_8_9_febrero',      # sáb–dom  8– 9 feb  (2 finde)
    'validaciones_10_16_febrero',    # lun–dom 10–16 feb  (5 lab + 2 finde)
    'validaciones_17_23_febrero',    # lun–dom 17–23 feb  (5 lab + 2 finde)
    'validaciones_24febrero_2marzo', # lun–dom 24feb–2mar (5 lab + 2 finde)
]

frames = []
for folder in OD_FOLDERS:
    p = MATRICES_DIR / folder / 'matriz_od.csv'
    df = pd.read_csv(p)
    frames.append(df)
    print(f'{folder}: {len(df):,} filas')

od = pd.concat(frames, ignore_index=True)
print(f'\nTotal filas concatenadas: {len(od):,}')
print('Módulo importado correctamente')

'validaciones_3_7_febrero'  'validaciones_8_9_febrero' 'validaciones_10_16_febrero' 'validaciones_17_23_febrero' 'validaciones_24febrero_2marzo'

---
## 1 — Carga de datos

In [ ]:
print(f'Filas: {len(od):,}')
print(f'Columnas: {list(od.columns)}')
od.head()

In [ ]:
n_origenes  = od['origen_punto_acceso'].nunique()
n_destinos  = od['destino_punto_acceso'].nunique()
n_estaciones = od[['origen_punto_acceso', 'destino_punto_acceso']].stack().nunique()
viajes_total = od['num_viajes'].sum()

print(f'Estaciones únicas (unión orig+dest): {n_estaciones:,}')
print(f'Orígenes únicos:                     {n_origenes:,}')
print(f'Destinos únicos:                     {n_destinos:,}')
print(f'Viajes totales:                      {viajes_total:,.0f}')
print(f'Intervalo mín: {od["intervalo"].min()}')
print(f'Intervalo máx: {od["intervalo"].max()}')

---
## 2 — Preparación temporal: slots y daytype

In [ ]:
# Instanciar modelo con parámetros de prueba
model = RegionFunctionLDA(n_topics=8, slot_minutes=15)

# Ejecutar _prepare internamente para inspeccionarlo
df_prep = model._prepare(od)

print('Columnas añadidas:', [c for c in df_prep.columns if c not in od.columns])
print()
print('Slots únicos:', sorted(df_prep['slot'].unique())[:10], '...')
print('Daytypes únicos:', df_prep['daytype'].unique())
print()
df_prep[['intervalo', 'slot', 'daytype']].head(8)

In [ ]:
# Verificar que los slots tienen sentido horario
# slot 0 = 00:00-00:14, slot 33 = 08:15-08:29, slot 96 = 24:00 (no debe existir)
slot_max = df_prep['slot'].max()
slot_min = df_prep['slot'].min()
slots_dia = 60 * 24 // model.slot_minutes

print(f'Slot mínimo: {slot_min}  → {slot_min * model.slot_minutes // 60:02d}:{slot_min * model.slot_minutes % 60:02d}h')
print(f'Slot máximo: {slot_max}  → {slot_max * model.slot_minutes // 60:02d}:{slot_max * model.slot_minutes % 60:02d}h')
print(f'Slots posibles por día con {model.slot_minutes}min: {slots_dia}')
assert slot_max < slots_dia, f'ERROR: slot {slot_max} fuera de rango (máx permitido {slots_dia-1})'
print('✓ Slots dentro de rango correcto')

In [ ]:
# Distribución de viajes por slot
viajes_por_slot = df_prep.groupby('slot')['num_viajes'].sum()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(viajes_por_slot.index, viajes_por_slot.values, color='steelblue', alpha=0.8)

# Etiquetas de hora cada 4 slots (1h)
tick_slots = range(0, slots_dia, 4)
ax.set_xticks(list(tick_slots))
ax.set_xticklabels(
    [f"{s * model.slot_minutes // 60:02d}h" for s in tick_slots],
    fontsize=8
)
ax.set_xlabel('Hora del día')
ax.set_ylabel('Viajes totales')
ax.set_title('Distribución de viajes por slot temporal')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

---
## 3 — Construcción de documentos

In [ ]:
import time
t0 = time.time()
docs = model.build_documents(od)
print(f'build_documents: {time.time()-t0:.1f}s')
print(f'Documentos generados (estaciones): {len(docs):,}')

In [ ]:
# Verificar formato de tokens
estacion_ejemplo = sorted(docs.keys())[0]
doc_ejemplo = docs[estacion_ejemplo]

print(f'Estación: {estacion_ejemplo}')
print(f'Tokens únicos en su documento: {len(doc_ejemplo):,}')
print()
print('Top 10 tokens por conteo:')
top10 = sorted(doc_ejemplo.items(), key=lambda x: -x[1])[:10]
for token, count in top10:
    print(f'  {token:<40}  {count:>8,.0f}')

In [ ]:
# Verificar que los tokens tienen el formato correcto: OUT/IN_<id>_k<NN>_WD/WE
# Los IDs de estación contienen guiones bajos (ej: 02_L3_P9) así que
# el patrón usa .+ en lugar de [^_]+ para no romper en los guiones del ID
import re
patron = re.compile(r'^(OUT|IN)_.+_k\d+_(WD|WE)$')

tokens_invalidos = 0
for estacion, doc in docs.items():
    for token in doc:
        if not patron.match(token):
            print(f'Token inválido: {token} en {estacion}')
            tokens_invalidos += 1

if tokens_invalidos == 0:
    print('✓ Todos los tokens tienen formato correcto')
else:
    print(f'✗ {tokens_invalidos} tokens con formato incorrecto')

In [ ]:
# Verificar que OUT e IN están presentes en los documentos
doc_con_out = sum(1 for doc in docs.values() if any(t.startswith('OUT') for t in doc))
doc_con_in  = sum(1 for doc in docs.values() if any(t.startswith('IN')  for t in doc))

print(f'Documentos con tokens OUT: {doc_con_out:,} / {len(docs):,}')
print(f'Documentos con tokens IN:  {doc_con_in:,}  / {len(docs):,}')

# Distribución del tamaño de documentos (nº de tokens únicos)
tam_docs = pd.Series({s: len(d) for s, d in docs.items()})
print()
print('Tokens únicos por documento:')
print(tam_docs.describe().round(1))

In [ ]:
# Histograma del tamaño de documentos
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(tam_docs.values, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_xlabel('Número de tokens únicos por estación')
ax.set_ylabel('Número de estaciones')
ax.set_title('Distribución del tamaño de documentos')
ax.axvline(tam_docs.median(), color='red', linestyle='--', label=f'Mediana: {tam_docs.median():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4 — Matriz documento-término (vocabulario)

In [ ]:
# Vectorizar para ver la matriz antes del LDA
X = model._vectorize_fit(docs)

print(f'Shape de X (estaciones × tokens): {X.shape}')
print(f'Densidad de la matriz:             {X.nnz / (X.shape[0] * X.shape[1]):.4%}')
print(f'Valores no cero:                   {X.nnz:,}')
print()

# Tamaño del vocabulario
vocab = model.vectorizer_.get_feature_names_out()
print(f'Vocabulario total (tokens únicos): {len(vocab):,}')

# Cuántos son OUT vs IN
n_out = sum(1 for t in vocab if t.startswith('OUT'))
n_in  = sum(1 for t in vocab if t.startswith('IN'))
print(f'  Tokens OUT: {n_out:,}')
print(f'  Tokens IN:  {n_in:,}')

# Cuántos son WD vs WE
n_wd = sum(1 for t in vocab if t.endswith('WD'))
n_we = sum(1 for t in vocab if t.endswith('WE'))
print(f'  Tokens WD:  {n_wd:,}')
print(f'  Tokens WE:  {n_we:,}  ← será 0 si solo hay un tipo de día en los datos')

---
## 5 — Ajuste LDA y validación de Theta

In [ ]:
import time

# ---------------------------------------------------------------------------
# Metro (02) — token_granularity='temporal'
#
# Vocabulario: 2 dirs × 7 franjas × 2 daytypes = 28 tokens.
#
# beta=1.0:
#   Con V=28 tokens y beta bajo (0.01, 0.1), la suma del prior Dirichlet
#   topic-word es muy pequeña → LDA crea tópicos "spike" donde 1 token acapara
#   ~100% del peso. Para que 6-7 tokens tengan peso real se necesita beta alto.
#   beta=1.0 → prior Dirichlet(1,...,1) uniforme sobre 28 tokens → los tópicos
#   se ven forzados a distribuirse sobre múltiples tokens, y las diferencias
#   entre estaciones son las que rompen la simetría para crear tópicos distintos.
#
# log_counts=False:
#   Con log1p las proporciones entre estaciones se comprimen (ratio 100:1 en
#   viajes pasa a ~2:1 tras log1p), reduciéndole señal a LDA para separar
#   perfiles residencial vs comercial vs hub. Con raw counts LDA recibe las
#   proporciones reales de viajes por franja, que es lo que diferencia las
#   funciones urbanas.
# ---------------------------------------------------------------------------
model_metro = RegionFunctionLDA(
    n_topics=10,
    slot_minutes=15,
    alpha=0.7,
    beta=1.0,
    max_iter=150,
    filter_prefixes=["02"],
    use_periods=True,       # 7 franjas → 28 tokens con temporal
    min_df=1,
    log_counts=False,
    token_granularity="temporal",
    n_init=3,
)
t0 = time.time()
theta_metro = model_metro.fit_transform(od)
print(f"Metro      → {theta_metro.shape}  ({time.time()-t0:.1f}s)")

# ---------------------------------------------------------------------------
# Cercanías (04) — mismos cambios de beta y log_counts
# ---------------------------------------------------------------------------
model_cercanias = RegionFunctionLDA(
    n_topics=8,
    slot_minutes=15,
    alpha=0.9,
    beta=1.0,
    max_iter=150,
    filter_prefixes=["04"],
    use_periods=True,       # 7 franjas → 28 tokens con temporal
    min_df=1,
    log_counts=False,
    token_granularity="temporal",
    min_station_trips=0,
    n_init=3,
)
t0 = time.time()
theta_cercanias = model_cercanias.fit_transform(od)
print(f"Cercanías  → {theta_cercanias.shape}  ({time.time()-t0:.1f}s)")

# Alias para compatibilidad con celdas siguientes
theta = theta_metro
model2 = model_metro

In [ ]:
# CHECK 1: cada fila debe sumar 1 en ambos modelos
for nombre, th in [('Metro', theta_metro), ('Cercanías', theta_cercanias)]:
    sumas = th.sum(axis=1)
    ok = ((sumas - 1.0).abs() < 1e-4).all()
    print(f'{nombre}: filas suman 1 → {"✓" if ok else "✗"}  '
          f'(min={sumas.min():.6f}, max={sumas.max():.6f})')


In [ ]:
# CHECK 2: valores en [0, 1]
for nombre, th in [('Metro', theta_metro), ('Cercanías', theta_cercanias)]:
    ok = th.values.min() >= 0 and th.values.max() <= 1
    print(f'{nombre}: valores en [0,1] → {"✓" if ok else "✗"}  '
          f'(min={th.values.min():.6f}, max={th.values.max():.6f})')


In [ ]:
# CHECK 3: diversidad de tópicos dominantes por red
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

for ax, (nombre, th) in zip(axes, [('Metro (02)', theta_metro), ('Cercanías (04)', theta_cercanias)]):
    # Ordenar numéricamente (no lexicográficamente: topic_0 < topic_1 < ... < topic_9 < topic_10)
    topic_cols = [f'topic_{i}' for i in range(th.shape[1])]
    dist = th.idxmax(axis=1).value_counts().reindex(topic_cols, fill_value=0)
    x = range(len(dist))
    ax.bar(x, dist.values, color='steelblue', alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels([f'T{i}' for i in range(len(dist))], fontsize=7)
    pct_max = dist.max() / len(th) * 100
    ax.set_xlabel('Tópico')
    ax.set_ylabel('Nº estaciones')
    ax.set_title(f'{nombre}\n({len(th)} estaciones, máx. concentración: {pct_max:.0f}%)')

plt.tight_layout()
plt.show()


In [ ]:
# Heatmap lado a lado: Metro vs Cercanías
nombres_orig = od.set_index('origen_punto_acceso')['origen_nombre'].to_dict()
nombres_dest = od.set_index('destino_punto_acceso')['destino_nombre'].to_dict()
nombres = {**nombres_orig, **nombres_dest}

def label(sid):
    n = nombres.get(sid, '')
    return f'{sid}  —  {n}' if n else sid

for titulo, th, model_r in [
    ('Metro (02)', theta_metro, model_metro),
    ('Cercanías (04)', theta_cercanias, model_cercanias),
]:
    th_sorted = th.sort_values('topic_0')
    n = len(th_sorted)
    fig, ax = plt.subplots(figsize=(11, max(5, n * 0.18)))
    im = ax.imshow(th_sorted.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
    ax.set_title(f'Theta — {titulo} ({n} estaciones, ordenadas por topic_0)')
    ax.set_xlabel('Tópico')
    ax.set_xticks(range(model_r.n_topics))
    ax.set_xticklabels([f'T{i}' for i in range(model_r.n_topics)], fontsize=9)
    ax.set_yticks(range(n))
    ax.set_yticklabels([label(s) for s in th_sorted.index], fontsize=6)
    plt.colorbar(im, ax=ax, label='Peso del tópico', shrink=0.6)
    plt.tight_layout()
    plt.show()
    print()


In [ ]:
# Tokens dominantes por tópico — Metro
print('=== METRO (02) ===')
print(model_metro.topic_summary(n_top_tokens=7))
print()
print('=== CERCANÍAS (04) ===')
print(model_cercanias.topic_summary(n_top_tokens=7))

In [ ]:
# Estación más representativa de cada tópico, por red
for nombre, th in [('Metro (02)', theta_metro), ('Cercanías (04)', theta_cercanias)]:
    print(f'=== {nombre} ===')
    for col in th.columns:
        best = th[col].idxmax()
        val  = th[col].max()
        nom  = nombres.get(best, '')
        print(f'  {col}: {best}  {nom}  ({val:.3f})')
    print()


---
## 6 — Inspección de tópicos

In [ ]:
import re

# Orden cronológico según el esquema de bins activo
_BIN_ORDER_7 = ['noche', 'madrug', 'punta_man', 'mediodia', 'tarde', 'punta_tar', 'noche_ini']
_BIN_ORDER_4 = ['night', 'morning', 'afternoon', 'evening']

def _get_bin_order(model_r):
    if model_r.use_periods == 4:
        return _BIN_ORDER_4
    elif model_r.use_periods is True:
        return _BIN_ORDER_7
    return None  # raw slots

def _extraer_bin(token, model_r):
    """
    Extrae (dirección, bin) del token.

    Con granularidad temporal los tokens son OUT_<franja>_(WD|WE) donde
    la franja puede contener '_' (punta_man, punta_tar, noche_ini).
    rsplit('_', 2) rompería en esos casos → se usa rsplit('_', 1) sobre
    el inner (sin el prefijo de dirección) para extraer siempre la franja
    completa independientemente de cuántos '_' contenga su nombre.
    """
    direccion = 'OUT' if token.startswith('OUT') else 'IN'
    # Quitar prefijo 'OUT_' o 'IN_' y el sufijo '_WD'/'_WE'
    inner = token[len(direccion) + 1:]      # ej: 'punta_man_WD'
    time_bin = inner.rsplit('_', 1)[0]      # ej: 'punta_man'
    if model_r.use_periods is False:
        # raw slots: time_bin es 'k31' → convertir a int
        try:
            time_bin = int(time_bin[1:]) if time_bin.startswith('k') else None
        except ValueError:
            time_bin = None
    return direccion, time_bin

# Distribución horaria de tokens dominantes por tópico — IN vs OUT
for titulo, model_r in [('Metro (02)', model_metro), ('Cercanías (04)', model_cercanias)]:
    topics_df = model_r.inspect_topics(n_top_tokens=30)
    n_topics  = model_r.n_topics
    bin_order = _get_bin_order(model_r)
    cols = min(4, n_topics)
    rows = (n_topics + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes = axes.flatten() if n_topics > 1 else [axes]

    for i, (topic, row) in enumerate(topics_df.iterrows()):
        bins_out, bins_in = {}, {}
        for token, peso in zip(row['top_tokens'], row['top_weights']):
            direccion, key = _extraer_bin(token, model_r)
            if key is None:
                continue
            if direccion == 'OUT':
                bins_out[key] = bins_out.get(key, 0) + peso
            else:
                bins_in[key]  = bins_in.get(key, 0) + peso

        ax = axes[i]
        if bin_order is not None:
            all_keys = [p for p in bin_order if p in bins_out or p in bins_in]
            x_pos = range(len(all_keys))
            out_vals = [bins_out.get(p, 0) for p in all_keys]
            in_vals  = [bins_in.get(p, 0)  for p in all_keys]
            w = 0.38
            ax.bar([x - w/2 for x in x_pos], out_vals, width=w, color='steelblue', alpha=0.85, label='OUT')
            ax.bar([x + w/2 for x in x_pos], in_vals,  width=w, color='darkorange', alpha=0.85, label='IN')
            ax.set_xticks(list(x_pos))
            ax.set_xticklabels(all_keys, rotation=45, ha='right', fontsize=6)
        else:
            slots_dia = 60 * 24 // model_r.slot_minutes
            all_keys = sorted(set(bins_out) | set(bins_in))
            out_vals = [bins_out.get(k, 0) for k in all_keys]
            in_vals  = [bins_in.get(k, 0)  for k in all_keys]
            ax.bar(all_keys, out_vals, color='steelblue', alpha=0.85, label='OUT')
            ax.bar(all_keys, in_vals,  color='darkorange', alpha=0.85, label='IN', bottom=out_vals)
            ax.set_xlim(0, slots_dia)

        ax.set_title(topic, fontsize=9)
        if i == 0:
            ax.legend(fontsize=6)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'Distribución horaria por tópico — {titulo}\n(azul=OUT, naranja=IN)', fontsize=11)
    plt.tight_layout()
    plt.show()
    print()

In [ ]:
# Versión compacta: solo model2 (metro), IN vs OUT separados
topics_df = model2.inspect_topics(n_top_tokens=30)
n_topics  = model2.n_topics
bin_order = _get_bin_order(model2)
cols = min(4, n_topics)
rows = (n_topics + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = axes.flatten() if n_topics > 1 else [axes]

for i, (topic, row) in enumerate(topics_df.iterrows()):
    bins_out, bins_in = {}, {}
    for token, peso in zip(row['top_tokens'], row['top_weights']):
        direccion, key = _extraer_bin(token, model2)
        if key is None:
            continue
        if direccion == 'OUT':
            bins_out[key] = bins_out.get(key, 0) + peso
        else:
            bins_in[key]  = bins_in.get(key, 0) + peso

    ax = axes[i]
    if bin_order is not None:
        all_keys = [p for p in bin_order if p in bins_out or p in bins_in]
        x_pos = range(len(all_keys))
        out_vals = [bins_out.get(p, 0) for p in all_keys]
        in_vals  = [bins_in.get(p, 0)  for p in all_keys]
        w = 0.38
        ax.bar([x - w/2 for x in x_pos], out_vals, width=w, color='steelblue', alpha=0.85, label='OUT')
        ax.bar([x + w/2 for x in x_pos], in_vals,  width=w, color='darkorange', alpha=0.85, label='IN')
        ax.set_xticks(list(x_pos))
        ax.set_xticklabels(all_keys, rotation=45, ha='right', fontsize=6)
    else:
        slots_dia = 60 * 24 // model2.slot_minutes
        all_keys = sorted(set(bins_out) | set(bins_in))
        out_vals = [bins_out.get(k, 0) for k in all_keys]
        in_vals  = [bins_in.get(k, 0)  for k in all_keys]
        ax.bar(all_keys, out_vals, color='steelblue', alpha=0.85, label='OUT')
        ax.bar(all_keys, in_vals,  color='darkorange', alpha=0.85, label='IN', bottom=out_vals)
        ax.set_xlim(0, slots_dia)

    ax.set_title(topic, fontsize=9)
    if i == 0:
        ax.legend(fontsize=6)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    'Distribución horaria de tokens dominantes por tópico — Metro\n(azul=OUT, naranja=IN)',
    fontsize=11,
)
plt.tight_layout()
plt.show()

In [ ]:
def plot_perfiles(estaciones, titulo_global, n_cols=None):
    """
    Dibuja el perfil de tópicos (distribución theta) de una lista de estaciones.

    Parameters
    ----------
    estaciones : list of (sid, theta_df, model, label)
    titulo_global : str
    n_cols : int or None — columnas del grid; si None usa len(estaciones)
    """
    n    = len(estaciones)
    cols = n_cols or n
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3.2), squeeze=False)
    axes = axes.flatten()

    for ax, (sid, th, model_r, titulo) in zip(axes, estaciones):
        if sid not in th.index:
            ax.set_title(f'{titulo}\n(no en theta)', fontsize=8, color='gray')
            continue
        perfil = th.loc[sid].values
        x = range(model_r.n_topics)
        ax.bar(x, perfil, color='steelblue', alpha=0.8)
        ax.set_ylim(0, max(perfil.max() * 1.15, 0.05))
        ax.set_xticks(list(x))
        ax.set_xticklabels([f'T{i}' for i in x], rotation=90, fontsize=5)
        ax.set_title(titulo, fontsize=9)

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(titulo_global, fontsize=11)
    plt.tight_layout()
    plt.show()


# Metro: SOL, NUEVOS MINISTERIOS, ATOCHA, AVENIDA DE AMERICA
plot_perfiles([
    ('02_L1_P12', theta_metro, model_metro, 'SOL (M · L1)'),
    ('02_L6_P18', theta_metro, model_metro, 'NUEVOS MIN. (M · L6)'),
    ('02_L1_P16', theta_metro, model_metro, 'ATOCHA (M · L1)'),
    ('02_L4_P11', theta_metro, model_metro, 'AV. DE AMERICA (M · L4)'),
], 'Perfil de tópicos — Metro: nodos centrales')


In [ ]:
# Cercanías: nodos centrales
plot_perfiles([
    ('04_L0_P66',  theta_cercanias, model_cercanias, 'RECOLETOS (C)'),
    ('04_L0_P108', theta_cercanias, model_cercanias, 'SOL (C)'),
    ('04_L0_P54',  theta_cercanias, model_cercanias, 'NUEVOS MIN. (C)'),
    ('04_L0_P7',   theta_cercanias, model_cercanias, 'MADRID ATOCHA (C)'),
], 'Perfil de tópicos — Cercanías: nodos centrales')


In [ ]:
# Cercanías: corredor C-4 norte (universidades y municipios)
plot_perfiles([
    ('04_L0_P29', theta_cercanias, model_cercanias, 'FUENCARRAL (C)'),
    ('04_L0_P10', theta_cercanias, model_cercanias, 'CANTOBLANCO UNIV. (C)'),
    ('04_L0_P98', theta_cercanias, model_cercanias, 'UNIV. P. COMILLAS (C)'),
    ('04_L0_P77', theta_cercanias, model_cercanias, 'TRES CANTOS (C)'),
    ('04_L0_P17', theta_cercanias, model_cercanias, 'COLMENAR VIEJO (C)'),
], 'Perfil de tópicos — Cercanías: universidades y corredor norte')


In [ ]:
# Cercanías: corredor C-2 este (Henares)
plot_perfiles([
    ('04_L0_P18',  theta_cercanias, model_cercanias, 'COSLADA (C)'),
    ('04_L0_P19',  theta_cercanias, model_cercanias, 'SAN FERNANDO (C)'),
    ('04_L0_P75',  theta_cercanias, model_cercanias, 'TORREJÓN DE ARDOZ (C)'),
    ('04_L0_P112', theta_cercanias, model_cercanias, 'SOTO DEL HENARES (C)'),
    ('04_L0_P106', theta_cercanias, model_cercanias, 'LA GARENA (C)'),
    ('04_L0_P1',   theta_cercanias, model_cercanias, 'ALCALÁ DE HENARES (C)'),
    ('04_L0_P2',   theta_cercanias, model_cercanias, 'ALCALÁ UNIV. (C)'),
    ('04_L0_P50',  theta_cercanias, model_cercanias, 'MECO (C)'),
    ('04_L0_P8',   theta_cercanias, model_cercanias, 'AZUQUECA (C)'),
    ('04_L0_P34',  theta_cercanias, model_cercanias, 'GUADALAJARA (C)'),
], 'Perfil de tópicos — Cercanías: corredor C-2 (Henares)', n_cols=5)

In [ ]:
# Metro L6: zona universitaria norte
plot_perfiles([
    ('02_L6_P21', theta_metro, model_metro, 'VICENTE ALEIXANDRE (M · L6)'),
    ('02_L6_P22', theta_metro, model_metro, 'CIUDAD UNIVERSITARIA (M · L6)'),
], 'Perfil de tópicos — Metro L6: zona universitaria norte')


In [ ]:
# Metro L8 completa — en dirección aeropuerto
# MAR DE CRISTAL es intercambiador L4↔L8; en el OD de la CRTM solo existe un
# punto_acceso registrado bajo L4 (02_L4_P18), no hay 02_L8_P4.
plot_perfiles([
    ('02_L8_P1', theta_metro, model_metro, 'NUEVOS MIN. (L8)'),
    ('02_L8_P2', theta_metro, model_metro, 'COLOMBIA (L8)'),
    ('02_L8_P3', theta_metro, model_metro, 'PINAR DEL REY (L8)'),
    ('02_L4_P18', theta_metro, model_metro, 'MAR DE CRISTAL (L4/L8)'),
    ('02_L8_P5', theta_metro, model_metro, 'FERIA DE MADRID (L8)'),
    ('02_L8_P6', theta_metro, model_metro, 'AEROPUERTO T1-T2-T3 (L8)'),
    ('02_L8_P7', theta_metro, model_metro, 'BARAJAS (L8)'),
    ('02_L8_P8', theta_metro, model_metro, 'AEROPUERTO T4 (L8)'),
], 'Perfil de tópicos — Metro L8', n_cols=4)


---
## 7 — Guardar Theta

In [ ]:
from pathlib import Path

out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

theta_metro.to_parquet(out_dir / 'theta_metro.parquet')
theta_metro.round(5).to_csv(out_dir / 'theta_metro.csv')
model_metro.inspect_topics(15).to_csv(out_dir / 'topics_metro.csv')

theta_cercanias.to_parquet(out_dir / 'theta_cercanias.parquet')
theta_cercanias.round(5).to_csv(out_dir / 'theta_cercanias.csv')
model_cercanias.inspect_topics(15).to_csv(out_dir / 'topics_cercanias.csv')

print(f'Metro      → theta_metro.parquet      {theta_metro.shape}')
print(f'Cercanías  → theta_cercanias.parquet  {theta_cercanias.shape}')
print(f'Guardado en: {out_dir.resolve()}')
